## Flow Cytometry Analysis — Experiment 2
### Author: Eleni Aretaki
### Date: 2026-03-24
### Purpose: Load, preprocess, normalize, and log2-transform flow cytometry data for Experiment 2.

### Experimental design - Similar as in Original FC - Additional conditions

#### 1. Petri Dishes per Condition:

- Each KO-drug (and WT-drug) combination is grown in its own dish. 
- DMSO is included as a control for both morning and afternoon treatment (same as above, WT-DMSO and KO-DMSO for all cell lines is grown in its own dish).
  
#### 2. 48h post Treatments:

 Treatments are applied and cells are harvested, measured 48h after treatment (just like in the original FC, where we had 48h).

#### 3. Four Replicates:

The experiment is repeated four times, which provides biological replicates for statistical analysis.

### Downstream Analysis 
#### Analysis of raw data has been performed in **R Studio** using packages for Flow Cytometry analysis, and NOT FlowJo. Relevant scripts can be found on GitHub.
#### Normalization and transformation follow the exact same set-up as in the first experiment, with the only difference being having no morning-afternoon distinction in the treatment time here.

##### In total, we have 6 compounds **(Niraparib, Cobimetinib, IlludinS and Lapatinib for all cell lines and Palbociclib, and MMS for specific ones only)** and 6 main cell lines **(ARID1A, ARID1B, ARID2, SMARCA4, BRD9, and WT)**.     
*SMARCC1, SMARCD1, ARID1B clone 2* and *WT + BRM014* are also screened for some drugs only. There is also one well where WT is treated with Camptothecin, which will not be used in our analysis, as no knockout was treated with the same drug, so there is no point in including it (experimental mistake - not part of the design).

In [ ]:
# -------------------------------
# Imports
# -------------------------------
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, levene
from statsmodels.stats.multitest import multipletests
import glob

In [ ]:
# -------------------------------
# File paths & constants
# -------------------------------
file_paths = [f"data/raw/Replicate_{i}_countdata.csv" for i in [1,2,3,5]]  # Replicates 1,2,3,5
annotation_file = "data/annotation_df2.csv"
out_dir = "results/experiment2/"
os.makedirs(out_dir, exist_ok=True)

phenotypes = [
    'S-phase',
    'G1-phase',
    'G2-phase',
    'Negative',
    'Apoptosis-Positive',
    'DSB-Positive',
    'Double-Positive'
]

# Mapping of complicated column names to simpler phenotype names
column_mapping = {
    '/Cells/Singlets/S': 'S-phase',
    '/Cells/Singlets/G1': 'G1-phase',
    '/Cells/Singlets/G2': 'G2-phase',
    '/Cells/Singlets/Negative': 'Negative',
    '/Cells/Singlets/Apoptosis-Positive': 'Apoptosis-Positive',
    '/Cells/Singlets/DSB-Positive': 'DSB-Positive',
    '/Cells/Singlets/Double-Positive': 'Double-Positive'
}

outliers = [
    (1, 'D3'), (1, 'A6'), (1, 'D4'), (1, 'A8'),
    (2, 'D3'), (2, 'A8'), (2, 'A6'), (2, 'D4'),
    (3, 'A8'), (3, 'D3'), (3, 'A6'), (3, 'D4'),
    (4, 'A8'), (4, 'D3'), (4, 'C12'), (4, 'B7'), (4, 'A6'), (4, 'D4')
]

ratio_phenotypes = ['S-phase', 'G1-phase', 'G2-phase']
difference_phenotypes = ['Apoptosis-Positive', 'DSB-Positive']

In [ ]:
# -------------------------------
# Functions
# -------------------------------

def load_annotation(file_path):
    """Load the annotation table."""
    return pd.read_csv(file_path, encoding='unicode_escape')

def load_and_merge_replicates(file_paths, annotation, column_mapping):
    """Load replicate CSVs, pivot percent_of_parent values, merge with annotation, and concatenate."""
    all_data = []
    for i, path in enumerate(file_paths, start=1):
        df = pd.read_csv(path, encoding='unicode_escape')
        percent_pivot = df.pivot(index="well", columns="Population", values="percent_of_parent")
        percent_pivot['replicate'] = i
        merged_df = pd.merge(percent_pivot, annotation, left_on="well", right_on="well_ID", how="inner")
        all_data.append(merged_df)
    combined_df = pd.concat(all_data, ignore_index=True)
    columns_to_keep = ["replicate", "well_ID", "modification", "treatment", "DMSO_conc"] + list(column_mapping.keys())
    combined_df = combined_df[columns_to_keep].rename(columns=column_mapping)
    combined_df['Apoptosis-Positive'] += combined_df['Double-Positive']
    combined_df['DSB-Positive'] += combined_df['Double-Positive']
    return combined_df

def remove_outliers(df, outliers, phenotypes):
    """Flag and remove outlier wells."""
    df = df.copy()
    df['include'] = True
    for pheno in phenotypes:
        df[pheno] *= 100
    for replicate, well in outliers:
        df.loc[(df['replicate'] == replicate) & (df['well_ID'] == well), 'include'] = False
    df = df[df['include']].drop(columns='include').reset_index(drop=True)
    return df

def normalize_data(df, ratio_phenotypes, difference_phenotypes):
    """Normalize treated wells using DMSO controls."""
    normalized_data = df.copy()
    rows_to_keep = []

    for experiment, group in df.groupby('replicate'):
        for mod in group['modification'].unique():
            mod_data = group[group['modification'] == mod]
            dmso_data = mod_data[mod_data['treatment'] == 'DMSO']

            if not dmso_data.empty:
                dmso_values = dmso_data.iloc[0]
                treated_mask = (
                    (normalized_data['replicate'] == experiment) &
                    (normalized_data['modification'] == mod) &
                    (normalized_data['treatment'] != 'DMSO')
                )
                for pheno in ratio_phenotypes:
                    normalized_data.loc[treated_mask, pheno] /= dmso_values[pheno]
                for pheno in difference_phenotypes:
                    normalized_data.loc[treated_mask, pheno] -= dmso_values[pheno]
                rows_to_keep.extend(normalized_data[treated_mask].index.tolist())
            else:
                print(f"⚠️ Excluding group: replicate {experiment}, modification {mod} — no DMSO control found.")

    return normalized_data.loc[rows_to_keep].reset_index(drop=True)

def log2_transform(df, ratio_phenotypes, difference_phenotypes):
    """Log2-transform ratio phenotypes, set negative difference phenotypes to 0."""
    transformed_df = df.copy()
    for pheno in ratio_phenotypes:
        transformed_df[pheno] = transformed_df[pheno].apply(lambda x: np.log2(x) if pd.notnull(x) and x > 0 else np.nan)
    for pheno in difference_phenotypes:
        transformed_df[pheno] = transformed_df[pheno].apply(lambda x: 0 if pd.notnull(x) and x < 0 else x)
    return transformed_df

def save_output(df, out_dir, filename):
    """Save dataframe to CSV."""
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(os.path.join(out_dir, filename), index=False)

In [ ]:
# -------------------------------
# Main pipeline
# -------------------------------
if __name__ == "__main__":
    annotation = load_annotation(annotation_file)
    combined_df = load_and_merge_replicates(file_paths, annotation, column_mapping)
    cleaned_df = remove_outliers(combined_df, outliers, phenotypes)
    normalized_df = normalize_data(cleaned_df, ratio_phenotypes, difference_phenotypes)
    log2_df = log2_transform(normalized_df, ratio_phenotypes, difference_phenotypes)

    # Save outputs
    save_output(cleaned_df, out_dir, "fc2_combined_data.csv")
    save_output(log2_df, out_dir, "fc2_normalized_data.csv")